In [13]:
import os
# in my pc set null
os.environ["CUDA_VISIBLE_DEVICES"] = ""
import torch
import numpy as np
from tqdm import tqdm

In [14]:
x_train = np.load("../dataset/mnist/x_train.npy")
y_train_label = np.load("../dataset/mnist/y_train_label.npy")
device = "cuda" if torch.cuda.is_available() else "cpu"


In [15]:
print(y_train_label[:10])

[5 0 4 1 9 2 1 3 1 4]


In [16]:
x = torch.tensor(y_train_label[:5], dtype=torch.int64, device=device)
y = torch.nn.functional.one_hot(x,  10)
print(y)

tensor([[0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
        [1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
        [0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 1]])


In [17]:
batch_size = 320
epochs = 1024

In [18]:
class MultilayerPerceptron(torch.nn.Module):
    def __init__(self):
        super().__init__()
        """
         tensor([[[ 1,  2],
          [ 3,  4]],

         [[ 5,  6],
          [ 7,  8]],

         [[ 9, 10],
          [11, 12]]])
        flatten to 3 * 4
        tensor([[ 1,  2,  3,  4],
         [ 5,  6,  7,  8],
         [ 9, 10, 11, 12]])
        """
        self.flatten = torch.nn.Flatten()
        self.linear_relu_stack = torch.nn.Sequential(
            # reduce dimension fron 28*28 to 512, then to 256, then to 128, finally to 10
            torch.nn.Linear(28*28, 512),
            torch.nn.ReLU(),
            torch.nn.Linear(512,256),
            torch.nn.ReLU(),
            torch.nn.Linear(256,128),
            torch.nn.ReLU(),
            torch.nn.Linear(128,10),
        )

    def forward(self, input):
        x = self.flatten(input)
        logits = self.linear_relu_stack(x)
        return logits



In [19]:
model = MultilayerPerceptron().to(device)
model = torch.compile(model, backend='aot_eager')

In [20]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [21]:
train_num = len(x_train) // batch_size
for epoch in range(20):
    train_loss = 0
    train_correct = 0
    for i in range(train_num):
        start = i * batch_size
        end = (i + 1 ) * batch_size
        train_batch = torch.tensor(x_train[start:end]).to(device)
        label_batch = torch.tensor(y_train_label[start:end], dtype=torch.int64).to(device)

        pred = model(train_batch)
        loss = loss_fn(pred, label_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_correct += (pred.argmax(1) == label_batch).type(torch.float32).sum().item()

    train_loss /= train_num
    print(f"epoch: {epoch}, loss: {train_loss}")
    accuracy = train_correct / (train_num * batch_size)
    print(f"epoch: {epoch}, accuracy: {accuracy}")



epoch: 0, loss: 0.47261503892626994
epoch: 0, accuracy: 0.8697860962566845
epoch: 1, loss: 0.16441229418079484
epoch: 1, accuracy: 0.9510026737967915
epoch: 2, loss: 0.10726368769923633
epoch: 2, accuracy: 0.9681316844919786
epoch: 3, loss: 0.07581204199854703
epoch: 3, accuracy: 0.977456550802139
epoch: 4, loss: 0.05672565187242898
epoch: 4, accuracy: 0.9826036096256684
epoch: 5, loss: 0.045723437461105576
epoch: 5, accuracy: 0.986062834224599
epoch: 6, loss: 0.04240621868670385
epoch: 6, accuracy: 0.986447192513369
epoch: 7, loss: 0.035973114479472615
epoch: 7, accuracy: 0.988485962566845
epoch: 8, loss: 0.025542422571284248
epoch: 8, accuracy: 0.9922627005347594
epoch: 9, loss: 0.020148542544052482
epoch: 9, accuracy: 0.993465909090909
epoch: 10, loss: 0.016255454055063505
epoch: 10, accuracy: 0.9948863636363636
epoch: 11, loss: 0.015120304236468676
epoch: 11, accuracy: 0.9949532085561498
epoch: 12, loss: 0.011305541838494373
epoch: 12, accuracy: 0.9961397058823529
epoch: 13, loss: 

In [22]:
torch.save(model, "mnist_mlp.pth")

Batch 1:
  calculate gradient grad_1
  zero_grad() → 0
  backward() → grad = grad_1
  step() → Adam use grad_1 to update also inter m,v

Batch 2:
  zero_grad() → 0（Adam m,v save！）
  cal grad_2
  backward() → grad = grad_2（new gradient）
  step() → Adam用grad_2 + previous m,v update

Adam （auto save）
m = β1 * m_prev + (1-β1) * grad  # first moment，direction
v = β2 * v_prev + (1-β2) * grad² # second moment，speed

┌─────────────────────────────────────────────────────┐
│  1. Forward:  输入 → 模型 → 预测值                   │
├─────────────────────────────────────────────────────┤
│  2. Loss:     预测值 + 真实标签 → loss值             │
│               ↓                                     │
│          loss_fn(pred, label)                       │
├─────────────────────────────────────────────────────┤
│  3. Backward: loss.backward()                       │
│               计算每个参数的梯度                     │
│               梯度存在 model.parameters().grad 中   │
├─────────────────────────────────────────────────────┤
│  4. Update:   optimizer.step()                      │
│               用梯度更新参数                         │
│               θ = θ - lr × gradient                 │
└─────────────────────────────────────────────────────┘


model.parameters() ←───┐
      │                │
      ▼                │
optimizer 持有引用 ─────┘

loss.backward()
      │
      ▼
把梯度写入 parameters.grad

optimizer.step()
      │
      ▼
读取 parameters.grad，更新 parameters


Loss	考试分数（告诉你差多少）
Loss.backward()	分析错误原因（计算梯度）
Optimizer	老师（根据错误调整学习方向）
Optimizer.step()	实际调整（更新参数